In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd

# Dummy data produksi
data = {
    'Date': ['2024-06-25', '2024-06-25', '2024-06-26', '2024-06-26', '2024-06-27', '2024-06-27'],
    'Shift': [1, 2, 1, 2, 1, 2],
    'Production_Line': ['A', 'A', 'B', 'B', 'C', 'C'],
    'Target_Units': [1000, 1000, 1200, 1200, 900, 900],
    'Actual_Units': [950, 900, 1150, 1100, 850, 800],
    'Downtime_Minutes': [20, 45, 10, 30, 15, 50],
    'Reject_Units': [15, 20, 5, 10, 8, 12]
}

df = pd.DataFrame(data)
print(df)


In [ ]:
# Tambah kolom Efficiency (%)
df['Efficiency'] = (df['Actual_Units'] / df['Target_Units']) * 100
print(df[['Date', 'Shift', 'Production_Line', 'Efficiency']])


Perhitungan Efficiency per Shift
✅ Insight:
Line mana yang performa-nya di bawah 95% bisa jadi prioritas improvement.

In [ ]:
# Tambah kolom Reject Rate (%)
df['Reject_Rate'] = (df['Reject_Units'] / df['Actual_Units']) * 100
print(df[['Date', 'Shift', 'Production_Line', 'Reject_Rate']])


Total Reject Rate
✅ Insight:
Line B lebih rendah reject-nya dibanding Line A & C, bisa jadi referensi SOP best practice.



In [ ]:
downtime_summary = df.groupby('Production_Line')['Downtime_Minutes'].sum().reset_index()
print(downtime_summary)


Total Downtime Per Line
✅ Insight:
Line mana yang paling sering downtime → bisa cek root cause maintenance.

In [ ]:
import matplotlib.pyplot as plt

df.groupby('Date')['Actual_Units'].sum().plot(kind='line', marker='o')
plt.title('Total Actual Production per Day')
plt.xlabel('Date')
plt.ylabel('Units Produced')
plt.grid(True)
plt.show()


Visualisasi Trend Produksi (Matplotlib)
✅ Insight:
Apakah output harian stabil atau ada tren turun.


In [ ]:
import pandas as pd

data_weekly = {
    'Date': ['2024-06-24', '2024-06-24', '2024-06-25', '2024-06-25', '2024-06-26', '2024-06-26'],
    'Shift': [1, 2, 1, 2, 1, 2],
    'Production_Line': ['A', 'A', 'B', 'B', 'C', 'C'],
    'Target_Units': [1000, 1000, 1200, 1200, 900, 900],
    'Actual_Units': [950, 900, 1150, 1100, 850, 800],
    'Downtime_Minutes': [20, 45, 10, 30, 15, 50],
    'Downtime_Reason': ['Machine Fail', 'Material Delay', 'Setup Change', 'Machine Fail', 'Power Cut', 'Machine Fail'],
    'Reject_Units': [15, 20, 5, 10, 8, 12],
    'Defect_Type': ['Scratch', 'Crack', 'Scratch', 'Bent', 'Crack', 'Scratch'],
    'Operator': ['Rudi', 'Dina', 'Bimo', 'Dina', 'Rudi', 'Bimo']
}

df_weekly = pd.DataFrame(data_weekly)
print(df_weekly)


In [ ]:
downtime_reason_summary = df_weekly.groupby('Downtime_Reason')['Downtime_Minutes'].sum().sort_values(ascending=False)
print(downtime_reason_summary)


Analisis 1: Pareto Downtime Reason
✅ Insight:
Downtime paling besar disebabkan oleh Machine Fail. Itu prioritas investigasi minggu ini.



In [ ]:
defect_summary = df_weekly['Defect_Type'].value_counts()
print(defect_summary)


Analisis 2: Defect Type Frequency
✅ Insight:
Scratch paling sering muncul → perlu cek SOP handling dan quality check.

In [ ]:
operator_reject = df_weekly.groupby('Operator')['Reject_Units'].sum().sort_values(ascending=False)
print(operator_reject)


Analisis 3: Operator Performance
✅ Insight:
Operator mana yang reject-nya paling tinggi → bisa kasih training tambahan.


In [ ]:
df_weekly['Variance'] = df_weekly['Actual_Units'] - df_weekly['Target_Units']
print(df_weekly[['Date', 'Production_Line', 'Actual_Units', 'Target_Units', 'Variance']])


Analisis 4: Actual vs Target Variance
✅ Insight:
Line mana yang consistently under target → perbaiki kapasitas/shift load.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Summarize total downtime per reason
downtime_reason_summary = df_weekly.groupby('Downtime_Reason')['Downtime_Minutes'].sum().sort_values(ascending=False)

# Plot bar chart
plt.figure(figsize=(8,5))
sns.barplot(x=downtime_reason_summary.values, y=downtime_reason_summary.index, palette='coolwarm')
plt.title('Total Downtime per Reason (Minutes)')
plt.xlabel('Total Downtime (Minutes)')
plt.ylabel('Downtime Reason')
plt.show()


Bar Chart Downtime per Reason

**Penjelasan:**

groupby() → grup data per downtime reason

sum() → jumlahkan total downtime tiap reason

sort_values() → urutkan dari terbesar

sns.barplot() → bikin horizontal bar chart

palette='coolwarm' → warna visualnya dari biru ke merah

In [ ]:
# Defect frequency
defect_summary = df_weekly['Defect_Type'].value_counts()

# Pie chart
plt.figure(figsize=(6,6))
plt.pie(defect_summary.values, labels=defect_summary.index, autopct='%1.1f%%', startangle=140)
plt.title('Defect Type Distribution (%)')
plt.axis('equal')  # biar pie chart-nya bulat
plt.show()


Pie Chart Defect Type Distribution

**Penjelasan**

value_counts() → hitung jumlah masing-masing defect type

plt.pie() → bikin pie chart

autopct → tampilkan persentase

startangle → sudut awal dari pie chart

axis('equal') → biar pie-nya bulat


In [ ]:
# Total reject per operator
operator_reject = df_weekly.groupby('Operator')['Reject_Units'].sum().sort_values(ascending=False)

# Bar chart
plt.figure(figsize=(7,5))
sns.barplot(x=operator_reject.index, y=operator_reject.values, palette='viridis')
plt.title('Total Reject per Operator')
plt.xlabel('Operator')
plt.ylabel('Total Reject Units')
plt.show()


Bar Chart Total Reject per Operator

**Penjelasan:**

groupby('Date') → totalin actual per hari

sns.lineplot() → bikin line chart

marker='o' → titik di tiap data point

grid(True) → tampilkan garis bantu


In [ ]:
# Total Production Summary
total_production = df_weekly['Actual_Units'].sum()
total_target = df_weekly['Target_Units'].sum()
total_variance = total_production - total_target
total_reject = df_weekly['Reject_Units'].sum()
total_downtime = df_weekly['Downtime_Minutes'].sum()

# Print Executive Summary
print("📑 Executive Summary Production Performance")
print("-------------------------------------------")
print(f"Total Production Target     : {total_target} Units")
print(f"Total Actual Production     : {total_production} Units")
print(f"Total Variance (Act - Tgt)  : {total_variance} Units")
print(f"Total Reject Units          : {total_reject} Units")
print(f"Total Downtime (Minutes)    : {total_downtime} Minutes")
print("\n")

# Top Downtime Reason
top_downtime = downtime_reason_summary.idxmax()
top_downtime_value = downtime_reason_summary.max()

print(f"🔍 Highest Downtime Reason  : {top_downtime} ({top_downtime_value} minutes)")

# Most Frequent Defect Type
top_defect = defect_summary.idxmax()
top_defect_value = defect_summary.max()

print(f"🔍 Most Frequent Defect     : {top_defect} ({top_defect_value} cases)")

# Highest Rejecting Operator
top_operator = operator_reject.idxmax()
top_operator_value = operator_reject.max()

print(f"🔍 Highest Reject by        : {top_operator} ({top_operator_value} units)")
print("-------------------------------------------")


Executive Summary Production Weekly Report

**Penjelasan:**

sum() → totalin semua produksi, target, downtime, reject

idxmax() → ambil nama/label dari nilai maksimum

max() → ambil nilai tertinggi-nya

print() → tampilkan di console kayak laporan rapat

In [ ]:
import pandas as pd

data_downtime = {
    'Date': ['2024-06-20', '2024-06-21', '2024-06-22', '2024-06-23', '2024-06-24'],
    'Downtime_Minutes': [50, 60, 55, 65, 80]
}

df_downtime = pd.DataFrame(data_downtime)
print(df_downtime)


In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Convert date to ordinal (angka hari)
df_downtime['Date_Ordinal'] = pd.to_datetime(df_downtime['Date']).map(pd.Timestamp.toordinal)

# X = Date as angka hari, y = Downtime
X = df_downtime[['Date_Ordinal']]
y = df_downtime['Downtime_Minutes']

# Model
model = LinearRegression()
model.fit(X, y)

# Predict next day downtime
next_day = pd.to_datetime('2024-06-25').toordinal()
predicted_downtime = model.predict([[next_day]])

print(f"📈 Predicted Downtime on 2024-06-25: {predicted_downtime[0]:.2f} Minutes")


Linear Regression (Simple Predictive Model)

**Penjelasan:**

Convert Date ke ordinal → biar jadi angka urut hari

LinearRegression() → model sederhana buat prediksi trend downtime

fit() → latih model-nya pakai histori

predict() → prediksi downtime tanggal berikutnya

**Insight:**
Downtime naik terus — prediksi besok tembus 82 menit.
Rekomendasi: lakukan inspeksi preventif hari ini sebelum jam produksi.